# Дни 4-5 на Kaggle: Qwen2.5-0.5B + LoRA против rubert-tiny2 и TF-IDF при одинаковом бюджете

Идея: три модели обучаются на **одной и той же подвыборке train** (N карточек, 1 эпоха для нейросетей) и оцениваются на полном validation. Так сравнивается сам подход, а не объём данных.

Перед запуском (Settings справа):
- Accelerator: **GPU T4 x2** (используется одна карта)
- Internet: **On** (скачивание Qwen с HuggingFace)
- Add Input: датасет с кодом (`src/` с `train_utils.py`, `project_utils.py`, `data_utils.py`; `scripts/` с `02`, `04`, `11`) и датасет со split (`split_ids.parquet`, `label_names.json`)

Порядок: 1) проверка скорости Qwen (smoke), 2) решить N по оценке времени, 3) базовые модели, 4) полный запуск Qwen (лучше через Save Version -> Save & Run All, с выключенным smoke).

In [ ]:
import os, sys, glob, shutil, subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HOME"] = "/kaggle/temp/hf"

def find_one(pattern, what):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f"Не найден {what}. Проверь Add Input."
    return hits[0]

SRC = os.path.dirname(find_one("/kaggle/input/**/data_utils.py", "src/data_utils.py"))
ROOT_IN = os.path.dirname(SRC)
for need in ("src/train_utils.py", "scripts/02_tfidf_baseline.py", "scripts/04_ablation_loss.py",
             "scripts/11_train_qwen_lora.py"):
    assert os.path.exists(f"{ROOT_IN}/{need}"), f"В датасете с кодом нет {need}: загрузи новую версию датасета."

PROJECT = "/kaggle/working/project"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(f"{PROJECT}/artifacts"); os.makedirs(f"{PROJECT}/results")
shutil.copytree(SRC, f"{PROJECT}/src")
shutil.copytree(f"{ROOT_IN}/scripts", f"{PROJECT}/scripts")
shutil.copy(find_one("/kaggle/input/**/split_ids.parquet", "split_ids.parquet"), f"{PROJECT}/artifacts/")
shutil.copy(find_one("/kaggle/input/**/label_names.json", "label_names.json"), f"{PROJECT}/artifacts/")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("Рабочая папка:", os.getcwd())

In [ ]:
!pip install -q peft
!pip list 2>/dev/null | grep -i -E "^(transformers|peft|datasets|torch|scikit-learn) "
!nvidia-smi -L

In [ ]:
N_TRAIN = 100_000        # размер общей подвыборки train; реши после smoke-замера
RUN_SMOKE = True         # проверка скорости Qwen (несколько минут)
RUN_BASELINES = False    # TF-IDF и rubert-tiny2 на тех же N карточках
RUN_QWEN = False         # основной запуск Qwen + LoRA

## 1. Проверка скорости Qwen

В конце скрипт печатает грубую оценку времени полного запуска. Если выходит больше 2.5 часов, уменьши `N_TRAIN`. Если в логе `loss: nan`, добавь `--fp32` (медленнее в 2-3 раза).

In [ ]:
if RUN_SMOKE:
    !python scripts/11_train_qwen_lora.py --n-train 2000 --val-n 1000 --tag smoke

## 2. Базовые модели при том же бюджете

TF-IDF (CPU) и rubert-tiny2 с weighted CE (lr 3e-4, 1 эпоха, seed 777) на тех же `N_TRAIN` карточках.

In [ ]:
if RUN_BASELINES:
    !python scripts/02_tfidf_baseline.py --max-train {N_TRAIN} --n-jobs 4
    !python scripts/04_ablation_loss.py --n-train {N_TRAIN} --epochs 1 --lr 3e-4 --arms weighted_ce --seeds 777 --out results/same_budget_tiny2.json

## 3. Основной запуск Qwen + LoRA

In [ ]:
if RUN_QWEN:
    !python scripts/11_train_qwen_lora.py --n-train {N_TRAIN} --tag n{N_TRAIN}

## 4. Итоговая таблица

In [ ]:
import json, os
rows = []
def load(p):
    return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else None

tf = load("results/tfidf_baseline_smoke.json")
if tf:
    rows.append(("TF-IDF + LinearSVC", tf["val"]["accuracy"], tf["val"]["f1_macro"], tf["seconds_total"], "CPU, включая подготовку"))
tn = load("results/same_budget_tiny2.json")
if tn and "weighted_ce_s777" in tn["runs"]:
    r = tn["runs"]["weighted_ce_s777"]
    rows.append(("RuBERT-tiny2, weighted CE", r["accuracy"], r["f1_macro"], r["train_seconds"], "T4, 1 эпоха"))
qw = load(f"results/qwen_lora_n{N_TRAIN}_val.json")
if qw:
    rows.append(("Qwen2.5-0.5B + LoRA, weighted CE", qw["val"]["accuracy"], qw["val"]["f1_macro"], qw["train_seconds"], "T4, 1 эпоха"))

print(f"Одинаковый бюджет: {N_TRAIN} карточек train, оценка на полном validation")
print(f"{'модель':<36}{'accuracy':>10}{'macro F1':>10}{'время, с':>10}  примечание")
for name, acc, f1, sec, note in rows:
    print(f"{name:<36}{acc:>10.4f}{f1:>10.4f}{sec:>10}  {note}")
if qw:
    print(f"\nQwen: обучаемых параметров {qw['params_trainable']:,} из {qw['params_total']:,}, "
          f"пик памяти {qw['peak_gpu_gb']} ГБ, инференс {qw['val']['predict_samples_per_s']} карточек/с")
print("\nОдин seed: разницу меньше ~1 п.п. по Macro F1 нельзя считать различием.")

In [ ]:
!ls -la results